# Causal Reasoning Deep Dive: Counterfactuals, Experiments, and Identification

**Topic 13 · 1 lecture**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb13_causal_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** **Causal reasoning**, the last and most demanding compass
position (the causal kind). Prediction was content to forecast *who*. Causation
insists on *because*, and that word has to be earned. You earn it with a
**counterfactual** (the outcome in the world you did not get to see) and an
**identification** argument (your written reason for trusting a comparison as
that missing world). Both get careful definitions below. One subtlety carries
over from the descriptive side: a causal claim still has a **reach**.
Identification buys the right to say *because* among your units. Carrying that
effect to a wider population is a separate crossing, bought with a sampling
argument (the sample effect SATE vs the population effect PATE, RDSS ch. 7).
That is the same silent-upgrade discipline the generalization position drills.

| | |
|---|---|
| **A claim this topic PERMITS** | "Because assignment was randomized (checked), or because nature randomized for me under a stated and probed assumption, the difference in outcomes carries a causal reading, with uncertainty." |
| **A claim this topic does NOT permit** | A causal claim whose counterfactual is *imagined* rather than *identified* ("they improved after the program, so the program worked"), or design vocabulary (DiD/RDD/IV) decorating a comparison whose assumptions were never argued. |

**Where this sits in the course:** Phase 4, pilot, diagnosis, and redesign. This
is the last of the four compass deep dives, and it closes the compass. It
develops milestone **M09 (Pilot Analysis)**, which you present as a four-minute
walkthrough at this week's Friday studio. The poster sprint starts the
following week.

*Provenance: RDSS ch. 6–7 (model/inquiry; potential outcomes; reach: SATE vs PATE) + ch. 18 'Experimental: causal' & declaration_18.1 (two-arm trial) + rdss foos_etal + ch. 16 'Observational: causal' + rdss cliningsmith_etal | the compass's causal kind at its reaches | two-arm-trial concept translated; real experiments reproduced descriptively; DiD/RDD/IV as identification arguments (intuition only) | translated (R→Python) & adapted*

## Learning Objectives

By the end of this notebook, you will be able to:

1. State the **counterfactual** at the heart of any causal claim using potential
   outcomes Y(1) and Y(0), and name the fundamental problem that keeps one of
   them always missing.
2. Explain how **random assignment** manufactures a valid comparison group, and
   write a three-sentence **identification argument**.
3. Analyze a real randomized field experiment, then say exactly what you
   computed and what you did not.
4. Read **DiD, RDD, and IV** as identification *arguments*, each with the one
   assumption a skeptic would attack first.
5. Write your own project's identification paragraph, or its honest absence,
   for the M09 pilot walkthrough.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters: the missing world

> *"Someone always tells me the patients who took the drug got better. Fine. So
> did the ones who rested, who prayed, who were younger, who were richer. 'They
> got better after X' is not 'X worked'; it is a story missing its comparison.
> Show me what would have happened WITHOUT X, or admit you are guessing."*
> — a **skeptical peer** who refuses to confuse *after* with *because*

Start with a scene you know. You take a pill and your headache fades an hour
later. Did the pill work? The honest answer needs the world where you skipped
the pill. That missing world is the **counterfactual**: the outcome that would
have happened under the choice not taken. Your headache after taking the pill
is a fact you observed. Your headache after *not* taking it is the
counterfactual, and it never happened.

Write this once in symbols, because the notation pays for itself. The
**treatment** is the thing being tested: the pill, the program, the policy. The
**control** condition is life without it, the comparison world. For one unit
(one person, one city), Y(1) is the outcome with the treatment and Y(0) is the
outcome without it. The causal effect *for that unit* is the difference,
Y(1) − Y(0): the world with the treatment minus the world without it.

Now the catch that governs the entire field, the **fundamental problem of
causal inference**: for any one unit you can only ever observe *one* of those
two worlds. A person either took the drug or did not. You never see both their
Y(1) and their Y(0). No amount of staring at the data a person *did* generate
recovers the outcome they *would* have had under the other condition.

There is one more villain to name before the rescue. A **confounder** is a
third factor that pushes on both who gets the treatment and the outcome.
Example: motivated people both show up to optional review sessions and study
harder on their own, so session-goers score higher even if the session taught
nothing. Confounders are why "the treated did better" proves so little on its
own.

### The rescue: random assignment

The escape is a trade, not magic. Give up on the *individual* effect and go
after the **average** effect across a group. Then engineer the data so one
group can honestly stand in for the other group's missing world. That
engineering is **random assignment**: letting pure chance, a coin flip or a
lottery, decide who gets the treatment. Example: a campaign flips a weighted
coin for each voter to decide who gets a door knock. Chance knows nothing about
motivation, age, or wealth. So the two groups it creates are alike on average
in every respect, measured and unmeasured, and no confounder can steer who
lands where. The control group's average outcome then honestly estimates what
the treated group *would* have done untreated.

**Identification** is the argument that connects the number you computed to the
causal claim you want to make. In plain terms: it is your written reason why
one group's outcomes can stand in for the other group's missing world. Example:
"assignment was random, so the arms are comparable." For an experiment, the
whole argument takes three sentences. (1) The quantity I want is the average
effect, Y(1) − Y(0), over my units. (2) Because treatment was assigned at
random, the control group is a valid counterfactual for the treated group.
(3) Therefore the difference in the two groups' average outcomes estimates the
causal effect, with uncertainty from sampling.

Drawn as a diagram, a clean experiment is almost boringly simple. That
simplicity is its whole power:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_18_1.png" width="60%"/></center>

*An experiment's causal diagram: a random assignment Z and a measurement
procedure Q both feed the outcome Y, and unknown heterogeneity U also affects
Y. Because Z is random, it is not tangled with U, and that is exactly what lets
the comparison carry a causal reading. (Figure from the replication materials
of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences*
(MIT-licensed archive; the book is free at book.declaredesign.org).)*

> **A question that often comes up here:** *"If we can never see both outcomes
> for one person, isn't causal inference just impossible?"* The individual
> effect is genuinely unrecoverable; that part really is impossible. What
> random assignment rescues is the **average** effect across a group, because
> chance makes the groups interchangeable. You trade "what did X do to *this*
> person?" for "what did X do *on average*?", and the second question has an
> honest answer.

---

### 🔮 Pause & Predict

The `foos_etal` file is a **real** UK get-out-the-vote field experiment. Chance
assigned some voters to a contact arm (the treatment: outreach from a
campaign). Others were left alone as the control arm. Whether each voter turned
out in 2014 (`marked_register_2014`) was recorded afterward. **Before running
the next cell**, commit to two predictions. Did being contacted *raise*
turnout? If so, by roughly how many percentage points? Write a direction and a
number.

### YOUR ANSWER HERE:

**Did contact raise turnout? (up / down / no change):**

**By roughly how many percentage points:**

**Why:**

---

### 🛠️ Run the Study: a real experiment, analyzed honestly

You will analyze the experiment with the simplest honest tool: the difference
in turnout between the treated arm and the control arm, plus its standard
error and a 95% interval. Two disciplines before the numbers appear.

- Say *exactly* what you compute. Here, an **unweighted, simple difference in
  means** across the two arms.
- Say *exactly* what you do not. The study behind this dataset ran a more
  elaborate, **weighted** analysis (the `weights` column is the clue). Our
  number reproduces the experiment's *logic*: randomize, then compare arms. It
  does not reproduce the paper's headline estimate, and claiming it did would
  be the failure.

> 💡 **Gemini Prompt:** "This code analyzes a real get-out-the-vote experiment as a simple unweighted difference in turnout between the contacted arm and the control arm, with a standard error for two proportions and a 95% interval: [paste the next cell]. Explain why randomization is what lets a plain difference in means carry a causal reading, and predict whether the interval will exclude zero."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed difference, its interval, and whether it excludes zero against Gemini's prediction, and report the sign honestly whichever way it comes out.
> - [ ] Confirm this is the UNWEIGHTED difference: be ready to say why it reproduces the experiment's logic, not the paper's weighted headline estimate.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the real experiment and confirm its shape before trusting anything.
foos = load_course_data("foos_etal.csv")
assert foos.shape == (8375, 5), "unexpected shape — flag this!"
print("✓ Loaded foos_etal.csv —", foos.shape[0], "rows,", foos.shape[1], "columns")
print()

# Group sizes and turnout by arm (UNWEIGHTED, simple).
arms = foos.groupby("treat")["marked_register_2014"].agg(n="count", turnout="mean")
arms.index = ["control (treat=0)", "contacted (treat=1)"]
print("Turnout by arm:")
print(arms.round(4))
print()

# Difference in means + its standard error (two proportions) + 95% interval.
t1 = foos.loc[foos.treat == 1, "marked_register_2014"]
t0 = foos.loc[foos.treat == 0, "marked_register_2014"]
diff = t1.mean() - t0.mean()
se = np.sqrt(t1.mean() * (1 - t1.mean()) / len(t1)
             + t0.mean() * (1 - t0.mean()) / len(t0))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se
print(f"difference (contacted - control): {diff:+.4f}  ({diff*100:+.1f} percentage points)")
print(f"standard error: {se:.4f}")
print(f"95% interval: [{ci_lo:+.4f}, {ci_hi:+.4f}]   (excludes 0: {ci_lo > 0})")

In [ ]:
# Self-check: the arms, the direction, and the interval are what we reported.
assert len(t1) == 6104 and len(t0) == 2271, "arm sizes changed — investigate before trusting"
assert diff > 0, "our estimate says contact RAISED turnout — report the sign honestly either way"
assert ci_lo > 0, "the 95% interval excludes zero"
print(f"✓ Self-check passed: contacted arm n={len(t1)}, control arm n={len(t0)}.")
print(f"✓ The unweighted difference in means is {diff*100:+.1f} points, "
      f"95% interval [{ci_lo*100:+.1f}, {ci_hi*100:+.1f}] points.")

### Balance: does the randomization look real?

Random assignment promises that the arms are alike *before* treatment. That
promise is checkable. If the arms differ sharply on something fixed before the
contact happened, the promise is in doubt. The cell below compares the arms on
a pre-treatment trait (ward) and on the study's `weights`.

> 💡 **Gemini Prompt:** "This code compares the two experimental arms on a pre-treatment trait (ward) and on the study's design weights: [paste the next cell]. Explain what a balance check is supposed to confirm about randomization, and why arms that line up on wards but differ sharply on weights would signal that treatment was assigned with UNEQUAL probabilities across groups."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed largest ward gap and the mean weight per arm: do the wards line up closely while the weights differ a lot, as Gemini describes?
> - [ ] Connect it to the estimate: be ready to say why differing weights make our simple unweighted difference a classroom approximation, not the paper's number.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Balance check: are the arms similar on pre-treatment features?
print("Mean design weight by arm:")
print(foos.groupby("treat")["weights"].mean().round(3).rename(
    {0: "control", 1: "contacted"}).to_string())
print()
ward_share = pd.crosstab(foos["ward"], foos["treat"], normalize="columns")
ward_share.columns = ["control", "contacted"]
gap = (ward_share["contacted"] - ward_share["control"]).abs()
print(f"Largest gap in any ward's share between arms: {gap.max():.3f}")
print("(wards line up fairly closely; the weights, however, differ a lot by arm)")

### 🔍 Reading the Evidence: write the three-sentence identification argument

The balance check reveals something honest and important. The `weights` differ
a lot across arms; the control arm carries much heavier weights. That is the
fingerprint of a design that assigned treatment with **unequal probabilities**
across groups and corrects for it with weights. Our simple unweighted
difference ignores that correction. That is exactly why it is a classroom
approximation, not the paper's estimate. An honest researcher reports the
method actually run and flags what a fuller analysis would change.

In the cell below, write the **three-sentence identification argument** for
this experiment: (1) the quantity you want, (2) why the control arm is a valid
counterfactual for the treated arm, (3) what therefore follows. Then add a
fourth sentence naming the one caveat our unweighted pass carries.

### YOUR ANSWER HERE:

**(1) The quantity I want:**

**(2) Why the control arm is a valid counterfactual:**

**(3) Therefore:**

**(4) The caveat my unweighted analysis carries:**

---

### 📝 Practice: name the counterfactual, then grade it

For each claim, write the **counterfactual it silently asserts** (the missing
world), then grade how plausibly that counterfactual is *identified*, low,
medium, or high, in one phrase.

- **A.** "People who attended the optional review session scored higher on the
  final, so the session raised scores."
- **B.** "The city cut its speed limit and crashes fell the next year, so the
  lower limit cut crashes."
- **C.** "In a coin-flip randomized trial, the group given the vaccine had fewer
  infections, so the vaccine reduced infections."


### YOUR ANSWER HERE:

**A (counterfactual / plausibility):**

**B (counterfactual / plausibility):**

**C (counterfactual / plausibility):**

---

## 2. Causal Claims Without Experiments

You have now seen the clean case. Here is the common one. **Guiding question:**
*when I cannot run an experiment, where might nature have run one for me, and
can I defend that it really did?*

> *"I don't get to randomize whether a country adopts a policy or whether a
> person lives through a war. Most questions I care about forbid the
> experiment. So my whole craft is finding the moments where the world
> randomized FOR me, and being brutally honest about whether it really did."*
> — the same **skeptical peer**, now working in the observational world

You usually cannot randomize. But sometimes the world does it for you. That is
a **natural experiment**: a situation where chance-like forces outside anyone's
control decided who got treated. Example: a visa lottery decides who makes a
pilgrimage; a birthday cutoff decides who starts school a year early. The
observational causal toolkit hunts for these moments, and each of its tools is
really an identification *argument* with one assumption you must defend, never
a formula you just apply.

### Three arguments, three pictures

- **Difference-in-differences (DiD).** Two groups, one treated, both tracked
  over time. If, absent treatment, they would have moved in **parallel**, then
  the treated group's *extra* movement after treatment is the effect. The
  assumption a skeptic attacks first: would the two groups really have moved
  together?
- **Regression discontinuity (RDD).** Treatment switches on at a sharp
  **threshold** of some score: a scholarship above a cutoff, a program below an
  income line. Units just below and just above the line are nearly identical
  except for treatment, so the *jump* in the outcome at the threshold is the
  effect. The attack: does anything else change at the cutoff, and can units
  nudge their own score onto the lucky side?
- **Instrumental variables (IV).** A **nudge** that pushes some units toward
  treatment but reaches the outcome *only through* treatment. If the nudge is
  as-good-as-random and has no other path, it isolates a causal effect. The
  attack: is the nudge truly random-like, with no back door to the outcome?

The intuition for each lives in a picture, and the optional section at the end
of this notebook draws two of them in simulated worlds you can run.

> **A question that often comes up here:** *"If DiD, RDD, and IV can get causal
> answers without an experiment, why bother randomizing at all?"* Because each
> of these buys its causal reading with an assumption a skeptic can attack:
> parallel trends, a clean unmanipulated cutoff, an instrument with no back
> door. Randomization makes the counterfactual valid *by construction*; the
> observational tools make it valid *by argument*, and arguments can be wrong.
> When the experiment is available, you take it.

### Design mimicry: the vocabulary is not the argument

One warning before you decide anything about your own project. The most common
causal failure among earnest researchers is **design mimicry**: borrowing the
vocabulary ("difference-in-differences," "regression discontinuity," "natural
experiment") to decorate a comparison whose assumptions were never argued.
Writing "DiD" does not make parallel trends true. Calling something a "natural
experiment" does not make the assignment random. The word *because* is earned
by the argued and probed assumption, never by the name of the method. A skeptic
reads past the label straight to the assumption, and so should you.

> **A question that often comes up here:** *"How do I know whether I have a
> real natural experiment or just a comparison I've labeled one?"* You
> interrogate the assignment, not the label. Ask the concrete question: was who
> got treated really as-good-as-random, and what specifically makes it so? Then
> state the one assumption a skeptic would attack and show the evidence that it
> holds. If the only thing making your comparison a "natural experiment" is
> that you called it one, you have the vocabulary and not the argument, and the
> word *because* has not been earned.

---

### ⚖️ Make a Design Choice: identify, or stop at association

Now turn the lens on your own project. In the cell below, choose honestly
between two options and defend the choice in a short paragraph.

- **(a) I can identify a causal effect.** Name the source of the counterfactual
  (a randomization you can run, or a natural experiment the world provides) and
  the assumption it rests on.
- **(b) I cannot identify a causal effect.** My honest claim stops at
  **association**: a pattern where two things move together, with no defended
  counterfactual behind it. I will word every finding that way ("X is
  associated with Y," never "X causes Y").

Choosing (b) is not a defeat. A precisely bounded associational claim is
publishable, honest research. A causal claim with no identification is a
defect.

> 💡 **Gemini Prompt:** "I am arguing that [my treatment] causes [my outcome] in my project. Here is my identification argument and its key assumption: [paste it]. Act as a skeptical reviewer: list the strongest threats to my counterfactual (reasons my treated and comparison groups might not be comparable) and, for each, name the evidence that would rule it out."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check each threat the AI raises against how treatment was *actually* assigned in your design; the AI cannot see your assignment mechanism.
> - [ ] Treat any "your design looks sound" reply as unearned reassurance, not a verdict. You confirm an assumption with evidence, never with an AI's approval.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**My choice (a / b):**

**Defense (the counterfactual source and its assumption, OR why I stop at association):**

---

### 🎯 Project Transfer: your identification paragraph and your walkthrough

Two artifacts leave this notebook with you, and both feed milestone **M09
(Pilot Analysis)**.

**First, the identification paragraph your project will carry**, or its honest
absence. If your design identifies an effect, state the quantity, the
counterfactual, and why the counterfactual is valid. If it does not, write the
honest version: "My design does not identify a causal effect; the
counterfactual is not available, so my claim stops at association because ___."

**Second, your pilot walkthrough.** At this week's Friday studio you present
your pilot in four minutes. Draft its three beats now: *what I ran / what came
out / what surprised me*. Then name the one verification step you most want a
listener to push you on. If your pilot is causal, its honesty lives in the
identification paragraph. If it is descriptive or predictive, state the claim
boundary your walkthrough will respect ("association, not cause"; "accuracy
checked on data the model never saw"). A pilot that *collapsed* during the week
is welcome: present the failure and your diagnosis, because that is legitimate,
gradable research. The poster sprint starts the following week, and it will
reuse every sentence you draft here.

### YOUR ANSWER HERE:

**My identification paragraph (or its honest absence):**

**What I ran:**

**What came out:**

**What surprised me:**

**The one verification step I want pushed on:**

---

### 🎟️ Claim Ticket

**Claim Ticket #1** — your project's causal status in one sentence: *"My
project's central claim is identified by [randomization / natural experiment /
___], or is honestly NOT identified, so it stops at association."*

### YOUR ANSWER HERE:

**My causal status, one sentence:**

---

## 3. If You Want to Go Further *(optional)*

Everything below is optional enrichment. Nothing in it is required for M09 or
the Friday studio. It shows two of the identification arguments actually
working in simulated worlds, then walks through a second real study: a natural
experiment built on a lottery.

### Seeing DiD work

The next cell draws a simulated world (seeded, not real data) built so the
parallel-trends assumption *holds*, with a known effect added after treatment
turns on. Watching the argument succeed teaches you what "it worked" looks
like, and therefore what a violation would look like in real data.

> 💡 **Gemini Prompt:** "This code simulates a difference-in-differences world where a treated and a control group would have moved in parallel absent treatment, then adds a known effect after treatment turns on: [paste the next cell]. Explain how DiD recovers the effect as the treated group's EXTRA movement beyond its parallel counterfactual, and predict that the printed post-treatment gap will equal the built-in effect."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that the printed post-treatment gap really matches the built-in effect the comment names; the picture should recover exactly what was planted.
> - [ ] Name the assumption this clean illustration hides: it is BUILT to have parallel trends, so be ready to say what a violation would look like in real data.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# ILLUSTRATION 1 (simulated, not data): a parallel-trends / DiD world.
did_rng = np.random.default_rng(SEED)
time = np.array([0, 1, 2, 3])
post = time >= 2                       # treatment turns on between period 1 and 2
control = 40 + 3 * time + did_rng.normal(0, 0.3, 4)
effect = 6                             # the built-in causal effect
treated_cf = control + 8               # counterfactual: parallel to control, shifted up
treated = treated_cf + effect * post   # treated = counterfactual PLUS the effect, post-treatment

fig, ax = plt.subplots()
ax.plot(time, control, "o-", color="#4C72B0", label="control group")
ax.plot(time, treated, "s-", color="#C44E52", label="treated group (observed)")
ax.plot(time, treated_cf, "s--", color="#C44E52", alpha=0.45,
        label="treated counterfactual (parallel, no treatment)")
ax.axvline(1.5, color="gray", linestyle=":", label="treatment turns on")
ax.set_xlabel("time period")
ax.set_ylabel("outcome")
ax.set_title("ILLUSTRATION: difference-in-differences — "
             "the gap between the red lines after treatment IS the effect")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

post_gap = (treated - treated_cf)[post]
print(f"Built-in effect = {effect}. Post-treatment gap (treated - its counterfactual) = "
      f"{np.round(post_gap, 2).tolist()}  → the picture recovers the effect.")

### Seeing RDD work

Same idea, different argument: a world where treatment switches on at a sharp
cutoff of a running score, with a known jump built in at the threshold.

> 💡 **Gemini Prompt:** "This code simulates a regression-discontinuity world where treatment switches on at a sharp cutoff of a running score, with a known jump at the threshold: [paste the next cell]. Explain why comparing units JUST below versus JUST above the cutoff identifies the effect, and why comparing points far apart on the running score would instead mix in the slope and mislead."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed local jump (means just below vs just above the cutoff) against the built-in jump: does the local comparison recover it?
> - [ ] Name the assumption the illustration hides: nothing else changes at the cutoff and units cannot precisely manipulate their score. Be ready to say why a real study must defend that.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# ILLUSTRATION 2 (simulated, not data): a threshold / RDD world.
rdd_rng = np.random.default_rng(SEED)
x = rdd_rng.uniform(0, 100, 300)       # a running score (e.g., an eligibility test)
cutoff = 50
jump = 8                               # the built-in effect: a jump at the cutoff
y = 20 + 0.3 * x + jump * (x >= cutoff) + rdd_rng.normal(0, 3, 300)

fig, ax = plt.subplots()
ax.scatter(x[x < cutoff], y[x < cutoff], s=14, color="#4C72B0", label="below cutoff (untreated)")
ax.scatter(x[x >= cutoff], y[x >= cutoff], s=14, color="#C44E52", label="at/above cutoff (treated)")
ax.axvline(cutoff, color="gray", linestyle=":", label=f"cutoff = {cutoff}")
ax.set_xlabel("running score")
ax.set_ylabel("outcome")
ax.set_title("ILLUSTRATION: regression discontinuity — "
             "the vertical CLIFF at the cutoff is the effect")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# A LOCAL comparison near the line recovers the jump (the far-apart means would
# mix in the slope, so we compare only points just below vs just above).
lo = (x >= 44) & (x < 50)
hi = (x >= 50) & (x < 56)
print(f"Built-in jump = {jump}. Just below the cutoff, mean outcome = {y[lo].mean():.1f}; "
      f"just above, {y[hi].mean():.1f}; local jump ≈ {y[hi].mean() - y[lo].mean():.1f}.")

The same threshold logic, drawn as a causal diagram, is what "RDD" actually
claims:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_16_8.png" width="60%"/></center>

*A regression-discontinuity diagram: a running variable X sets a cutoff-based
treatment D, while X and unknown heterogeneity U both feed the outcome Y. Only
right at the cutoff, where X barely differs on either side, does the jump in D
cleanly reveal its effect. (Figure from the replication materials of Blair,
Coppock & Humphreys (2023), *Research Design in the Social Sciences*
(MIT-licensed archive; the book is free at book.declaredesign.org).)*

### A real natural experiment: the Hajj lottery

Now leave the illustrations for real data. The `cliningsmith_etal` file is the
replication dataset behind the **Clingingsmith, Khwaja & Kremer** Hajj-lottery
study. Among people who *applied* to make the pilgrimage, a lottery decided who
received a visa (`success` = 1 for lottery winners). Because the lottery is
random *among applicants*, winners and losers are comparable. The world ran the
randomization for us: a textbook natural experiment.

The `views_*` columns record attitudes toward other national and ethnic groups,
and `views` is a composite index of them. The cell below computes the **simple
mean difference** in `views` between lottery winners and losers. Again this is
the classroom-simple, unadjusted version, presented with its
lottery-as-randomization argument.

> 💡 **Gemini Prompt:** "This code computes the simple unadjusted mean difference in a composite 'views toward other groups' index between people who WON a visa lottery and those who lost, among applicants to the Hajj pilgrimage: [paste the next cell]. Explain why a lottery among applicants makes winners and losers comparable (a natural experiment), and why that limits the causal claim to applicants rather than to everyone."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed difference and its interval: does it exclude zero, and does the sign match the direction you expected the pilgrimage to move views?
> - [ ] Confirm the claim's boundary: the lottery randomizes among applicants only, so be ready to say who this result does and does not speak for.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the natural experiment and verify its columns/means before claiming anything.
hajj = load_course_data("cliningsmith_etal.csv")
assert hajj.shape == (958, 8), "unexpected shape — flag this!"
print("✓ Loaded cliningsmith_etal.csv —", hajj.shape[0], "rows,", hajj.shape[1], "columns")
print("Columns:", list(hajj.columns))
print()

views = hajj.groupby("success")["views"].agg(n="count", mean="mean")
views.index = ["lost lottery (success=0)", "won lottery (success=1)"]
print("Composite 'views toward other groups' by lottery outcome:")
print(views.round(4))
print()

w1 = hajj.loc[hajj.success == 1, "views"]
w0 = hajj.loc[hajj.success == 0, "views"]
d = w1.mean() - w0.mean()
se = np.sqrt(w1.var(ddof=1) / len(w1) + w0.var(ddof=1) / len(w0))
print(f"difference (won - lost): {d:+.4f}   SE: {se:.4f}   "
      f"95% interval: [{d - 1.96*se:+.4f}, {d + 1.96*se:+.4f}]")

In [ ]:
# Self-check: direction and interval match what we will claim.
assert hajj.shape == (958, 8)
assert w1.mean() > w0.mean(), "winners' mean should exceed losers' — report the sign honestly"
assert d - 1.96 * se > 0, "the 95% interval excludes zero"
print(f"✓ Self-check passed: lottery winners score {d:+.2f} higher on the composite "
      "views index, interval excluding zero.")
print("✓ This is a simple unadjusted difference — the classroom-simple version.")

### 🔍 Reading the Evidence: what the lottery lets you claim

In the cell below, answer two things about the roughly +0.47 difference.
First: what is the **most precise causal claim** the lottery permits, and over
*whom*? Careful: the lottery randomizes among *applicants*, not among all
people. Second: name two things this difference does **not** establish, no
matter how clean the lottery. For instance, the specific mechanism, or that
our simple unadjusted number equals the study's full estimate.

### YOUR ANSWER HERE:

**The most precise causal claim the lottery permits (and over whom):**

**Two things it does NOT establish:**

---

### 📝 Practice: match the design, then name the first attack

Match each scenario to its identification argument (DiD, RDD, or IV, each used
once), and write the **one sentence a skeptic would attack first**: the
assumption the whole claim rests on.

- **A.** "One state raised its minimum wage in 2015; a neighboring state did not.
  We compare each state's change in employment from 2014 to 2016."
- **B.** "A merit scholarship goes to applicants scoring at or above 1200 on a
  test; we compare college enrollment for applicants just below versus just above
  1200."
- **C.** "Rainfall on election day nudges some people to stay home; we use
  election-day rainfall to study how turnout affects which candidate wins."


### YOUR ANSWER HERE:

**A (design / first attack):**

**B (design / first attack):**

**C (design / first attack):**

---

## 4. Wrap-Up

Today you learned what it takes to earn the word *because*. You met the
**counterfactual**, the missing world Y(0) or Y(1) you never see for one unit,
and watched random assignment rescue the *average* effect by making one group a
valid stand-in for another's missing world. You analyzed a **real**
get-out-the-vote experiment honestly: a +3.4-point unweighted difference with
an interval excluding zero, flagged openly as the classroom-simple version of a
weighted study. And you met the world's own experiments: the parallel lines of
DiD, the cliff of RDD, the nudge of IV, each an *argument* whose one assumption
a skeptic attacks first.

> **"The word 'because' is not earned by a method's name — it is earned by a
> counterfactual you can defend."**

This closes the compass: all four positions are now lit. Your identification
paragraph and your three walkthrough beats travel with you to this week's
Friday studio for the M09 pilot walkthrough. The following week the course
turns from finding evidence to **communicating** it: the poster sprint opens,
and everything you have built (a claim, a compass position, an honest boundary)
has to survive a stranger's ninety-second read. Bring your best figure and your
headline claim.

---

## 5. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 6–7 (model/inquiry; potential outcomes; reach: SATE vs PATE) | the sample-vs-population reach note within the causal kind | conceptual crosswalk to the generalization position | adapted*
- *RDSS ch. 18 'Experimental: causal' + declaration_18.1 (two-arm trial) | experimental causal designs | two-arm-trial concept; real GOTV experiment reproduced as an unweighted difference in means | translated (R→Python) & adapted*
- *RDSS ch. 16 'Observational: causal' | observational causal designs | DiD/RDD/IV as identification arguments, intuition only, no estimation machinery | adapted*
- *fresh | two seeded illustrative simulations (parallel-trends world, threshold world) | — | fresh*
- *foos_etal.csv | rdss package data | Foos et al. UK get-out-the-vote field experiment replication; unweighted classroom analysis (the study's own analysis is weighted) | adapted*
- *cliningsmith_etal.csv | rdss package data | Clingingsmith, Khwaja & Kremer Hajj-lottery study replication; simple unadjusted mean difference | adapted*

**Dataset attribution:** Datasets from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 7 'Defining the inquiry' (potential outcomes; reach: SATE vs
  PATE), ch. 16 'Observational: causal', and ch. 18 'Experimental: causal'.
  Free online: [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>